In [ ]:
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import os

In [ ]:
# Cargar conjuntos estratificados
X_train_cls = np.load('data/X_train_class.npy')
X_test_cls = np.load('data/X_test_class.npy')

# Binarizar (redondear) los valores de píxeles
X_train_cls_bin = np.round(X_train_cls)
X_test_cls_bin = np.round(X_test_cls)

# Confirmar las formas y tipos
print("X_train_cls_bin shape:", X_train_cls_bin.shape)
print("X_test_cls_bin shape:", X_test_cls_bin.shape)
print("Valores únicos en X_train:", np.unique(X_train_cls_bin))

### Función para agregar ruido

In [ ]:
def agregar_ruido_vario(imagenes, radius=10, num_patches=2):
    imagenes_con_ruido = imagenes.copy()
    h, w = imagenes.shape[1:3]

    for i in range(imagenes.shape[0]):
        for _ in range(num_patches):
            centro_x = np.random.randint(radius, w - radius - 1)
            centro_y = np.random.randint(radius, h - radius - 1)
            patch = imagenes[i, centro_y - radius:centro_y + radius + 1,
                               centro_x - radius:centro_x + radius + 1, 0]
            if np.any(patch < 1.0):
                imagenes_con_ruido[i, centro_y - radius:centro_y + radius + 1,
                                      centro_x - radius:centro_x + radius + 1, 0] = 1.0
    return imagenes_con_ruido


In [ ]:
# Prueba en memoria: no guarda ni modifica archivos
X_prueba = X_train_cls_bin[:2].copy()

X_ruido_prueba = agregar_ruido_vario(
    X_prueba,
    radius=10,
    num_patches=2
)

print("Dimensiones:", X_ruido_prueba.shape)
print("Valores:", np.unique(X_ruido_prueba))
print("Píxeles modificados:", np.count_nonzero(X_prueba != X_ruido_prueba))

### Prueba 1

In [ ]:
train_ruido_10_2 = agregar_ruido_vario(X_train_cls_bin, radius=10, num_patches=2)
test_ruido_10_2 = agregar_ruido_vario(X_test_cls_bin, radius=10, num_patches=2)


In [ ]:
n = 10  # Número de imágenes a mostrar
plt.figure(figsize=(20, 4))

for i in range(n):
    # Imagen original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_train_cls_bin[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis('off')

    # Imagen con ruido
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(train_ruido_10_2[i].squeeze(), cmap='gray')
    plt.title("Con ruido")
    plt.axis('off')

plt.tight_layout()
plt.show()


### Generación de imágenes ruidosas

In [ ]:
# Directorio de salida
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Definir parámetros
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Generar y guardar combinaciones
for num_patches in num_patches_list:
    for radius in radius_list:
        print(f"Generando ruido con {num_patches} parches de radio {radius}...")

        train_ruido = agregar_ruido_vario(X_train_cls_bin, radius=radius, num_patches=num_patches)
        test_ruido = agregar_ruido_vario(X_test_cls_bin, radius=radius, num_patches=num_patches)

        nombre_train = f"train_ruido_{num_patches}_{radius}.npy"
        nombre_test = f"test_ruido_{num_patches}_{radius}.npy"

        np.save(os.path.join(output_dir, nombre_train), train_ruido)
        np.save(os.path.join(output_dir, nombre_test), test_ruido)

print("Todos los conjuntos ruidosos han sido generados y guardados.")


In [ ]:
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Mostrar 10 imágenes comparativas por combinación
n = 10

for num_patches in num_patches_list:
    for radius in radius_list:
        nombre_train = f"train_ruido_{num_patches}_{radius}.npy"
        path = os.path.join(output_dir, nombre_train)

        # Cargar conjunto ruidoso
        imgs_ruido = np.load(path)

        # Mostrar comparaciones con el conjunto original binarizado
        plt.figure(figsize=(20, 4))
        for i in range(n):
            # Original
            ax = plt.subplot(2, n, i + 1)
            plt.imshow(X_train_cls_bin[i].squeeze(), cmap='gray')
            plt.title("Original")
            plt.axis('off')

            # Con ruido
            ax = plt.subplot(2, n, i + 1 + n)
            plt.imshow(imgs_ruido[i].squeeze(), cmap='gray')
            plt.title("Con ruido")
            plt.axis('off')

        plt.suptitle(f"Ruido: {num_patches} parches de radio {radius}", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.93])  # espacio para el título
        plt.show()
